# From Self-RAG to Agentic RAG: An Enterprise Case Study

**Objective:** Learn when and why to upgrade from Self-RAG to Agentic RAG

## 1. Business Case: Insurance Claims Processing

**Acme Insurance Corp** processes 50,000+ claims monthly.

### Business Problem (5 Key Points)

1. **High Volume, Complex Queries**: Questions span multiple documents
2. **Time-Critical Decisions**: 3 days spent gathering information
3. **Accuracy Requirements**: $2.3M annual cost from incorrect decisions
4. **Knowledge Fragmentation**: 15,000+ documents across systems
5. **Compliance Pressure**: Audit trails required with source citations

## 2. Setup

In [1]:
# Install packages
# Chat/LLM calls use Groq via its OpenAI-compatible API.
# Groq has no embeddings endpoint, so embeddings run locally via sentence-transformers.
# opik adds LLM observability/tracing (Comet's open-source platform).
%pip install -q openai langchain-openai langchain-huggingface sentence-transformers langchain-chroma langchain-core chromadb python-dotenv opik

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Load environment
import os
from dotenv import load_dotenv
import getpass
load_dotenv()

# Groq serves an OpenAI-compatible API, so we reuse the OpenAI SDK / langchain-openai
# clients and just point them at Groq's base URL with a Groq API key (gsk_...).
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GROQ_CHAT_MODEL = "openai/gpt-oss-120b"

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API Key:")
if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY not found in environment or .env file")

print(f"API Key loaded: ...{os.getenv('GROQ_API_KEY')[-4:]}")

# --- Opik observability config ---
# Reads OPIK_API_KEY / OPIK_WORKSPACE from .env (get a key at
# https://www.comet.com/api/my/settings). Mirrors the GROQ_API_KEY fallback above.
import opik

if not os.getenv("OPIK_API_KEY"):
    os.environ["OPIK_API_KEY"] = getpass.getpass("Opik API Key:")

OPIK_PROJECT_NAME = "agentic-rag-insurance"   # all traces group under this project

opik.configure(
    api_key=os.environ["OPIK_API_KEY"],
    workspace=os.getenv("OPIK_WORKSPACE"),    # None -> uses "default"
    # use_local=True would target a self-hosted Opik instead
)
print(f"Opik configured -> project '{OPIK_PROJECT_NAME}'")

API Key loaded: ...P5KV


OPIK: You already have an API key set in the configuration file. If you want to change it, please use the --force flag or force=True when calling the configure() method. Otherwise, the configuration file will not be updated but the session will use the new API key.
OPIK: Opik is already configured. You can check the settings by viewing the config file at C:\Users\preet\.opik.config
OPIK: Configuration completed successfully. Traces will be logged to 'Opik Demo Agent Observability' project. To change the destination project, see: https://www.comet.com/docs/opik/tracing/log_traces#configuring-the-project-name
OPIK: Pre-fetching the Opik MCP server (uv tool install opik-mcp)...
OPIK: Claude Code: Registered 'opik-mcp' via `claude mcp add` (user scope)
OPIK: Restart your AI host to pick up the Opik MCP server, then ask it to 'list my Opik projects' to confirm the connection.


Opik configured -> project 'agentic-rag-insurance'


In [4]:
# Sample enterprise documents
INSURANCE_DOCUMENTS = [

    # Defines coverage limits, deductibles, exclusions, and claim deadlines for auto insurance policies
    {
        "id": "policy_auto_001",
        "content": """AUTO INSURANCE POLICY TERMS - COMPREHENSIVE COVERAGE

Coverage includes:
- Collision damage: Up to $50,000 per incident
- Theft protection: Full replacement value
- Natural disaster damage: Covered
- Deductible: $500 standard, $1000 for high-risk drivers

Exclusions:
- Racing or competitive events
- Commercial use without rider
- Intentional damage

Claim filing deadline: 30 days from incident.""",
        "metadata": {"type": "policy", "category": "auto"}
    },

    # Specifies coverage limits, exclusions, and special conditions for home insurance policies
    {
        "id": "policy_home_001",
        "content": """HOME INSURANCE POLICY TERMS - STANDARD COVERAGE

Coverage includes:
- Dwelling coverage: Up to $500,000
- Personal property: Up to $250,000
- Liability protection: $100,000 per occurrence

Water damage conditions:
- Burst pipes: Covered if sudden failure
- Flood damage: NOT covered (requires separate policy)
- Sewer backup: Covered with optional rider

Claim filing deadline: 60 days from discovery.""",
        "metadata": {"type": "policy", "category": "home"}
    },

    # Provides internal operational rules for claim prioritization, approvals, documentation, and escalation
    {
        "id": "guideline_claims_001",
        "content": """CLAIMS PROCESSING GUIDELINES - VERSION 3.2

Priority Processing Rules:
- Claims over $10,000: Require supervisor approval
- Claims with injuries: Process within 48 hours
- Multiple claims same policy: Flag for fraud review

Documentation Requirements:
- Photo evidence for property damage
- Police report for theft claims
- Repair estimates from 2 contractors

Escalation: Analyst -> Senior Analyst -> Claims Manager -> VP Claims""",
        "metadata": {"type": "guideline", "category": "claims"}
    },

    # Captures legally mandated claim handling timelines and consumer protection requirements for California
    {
        "id": "regulation_ca_001",
        "content": """STATE REGULATORY REQUIREMENTS - CALIFORNIA

Claims Processing Timelines:
- Acknowledge claim receipt: Within 15 days
- Accept or deny claim: Within 40 days
- Payment after acceptance: Within 30 days

Consumer Protection:
- Must provide written denial explanation
- Must inform of appeal rights

Penalties: Up to $10,000 per violation""",
        "metadata": {"type": "regulation", "state": "CA"}
    },

    # Records historical claims activity and risk assessment for a specific policyholder (John Smith)
    {
        "id": "claims_history_001",
        "content": """CLAIMS HISTORY - John Smith (Policy #AC-2024-78432)

Claim 1 (2023-03-15): Fender bender, $2,340 paid
Claim 2 (2023-08-22): Windshield replacement, $450 paid
Claim 3 (2024-01-10): Catalytic converter theft, $1,800 paid

Risk Assessment: Medium (3 claims in 12 months)
Premium Adjustment: +15% at renewal
Fraud Indicators: None detected""",
        "metadata": {"type": "claims_history", "policyholder": "John Smith"}
    },

    # Defines step-by-step operational procedures for assessing and processing water damage claims
    {
        "id": "procedure_water_001",
        "content": """WATER DAMAGE CLAIMS - SPECIAL PROCEDURES

Step 1: Determine water source
- Category 1 (Clean): Broken supply lines
- Category 2 (Gray): Appliance overflow
- Category 3 (Black): Sewage, flooding

Step 2: Coverage assessment
- Sudden/accidental: Typically covered
- Gradual damage: NOT covered
- Maintenance issues: NOT covered

Required: Photos, plumber report, moisture readings
Processing time: 14-21 days""",
        "metadata": {"type": "procedure", "category": "water_damage"}
    }
]

print(f"Created {len(INSURANCE_DOCUMENTS)} documents")


Created 6 documents


In [5]:
# Setup vector store
# Groq does not offer an embeddings endpoint, so we embed locally with a small
# sentence-transformers model (runs on CPU, downloaded once on first use).
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

documents = [
    Document(page_content=doc["content"], metadata=doc["metadata"])
    for doc in INSURANCE_DOCUMENTS
]

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="acme_insurance"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"Vector store ready with {len(documents)} documents")

c:\Users\preet\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2924.66it/s]


Vector store ready with 6 documents


## 3. Self-RAG Implementation

In [6]:
from langchain_openai import ChatOpenAI
from opik import track
from opik.integrations.langchain import OpikTracer

class SelfRAG:
    def __init__(self, retriever):
        self.retriever = retriever
        # langchain-openai's ChatOpenAI talks to Groq's OpenAI-compatible endpoint.
        self.llm = ChatOpenAI(
            model=GROQ_CHAT_MODEL,
            temperature=0,
            base_url=GROQ_BASE_URL,
            api_key=os.environ["GROQ_API_KEY"],
        )

        # Opik callback: traces every LangChain chain.invoke() below as nested spans.
        self.opik_tracer = OpikTracer(tags=["self-rag"])

        self.relevance_prompt = ChatPromptTemplate.from_template("""
Is this document relevant to the query? Answer only RELEVANT or NOT_RELEVANT.

Query: {query}
Document: {document}
""")

        self.generation_prompt = ChatPromptTemplate.from_template("""
You are a claims analyst. Answer using ONLY the provided context.

Context:
{context}

Query: {query}

Answer with specific references to sources.
""")

    @track(project_name=OPIK_PROJECT_NAME)
    def query(self, query: str) -> dict:
        print(f"\n{'='*60}")
        print(f"SELF-RAG: {query[:50]}...")
        print(f"{'='*60}")

        # Retrieve
        print("\n1. Retrieving...")
        docs = self.retriever.invoke(query)
        print(f"   Found {len(docs)} documents")

        # Filter relevant
        print("\n2. Checking relevance...")
        relevant_docs = []
        for i, doc in enumerate(docs):
            chain = self.relevance_prompt | self.llm | StrOutputParser()
            result = chain.invoke(
                {"query": query, "document": doc.page_content[:500]},
                config={"callbacks": [self.opik_tracer]},
            )
            is_relevant = "RELEVANT" in result.upper() and "NOT" not in result.upper()
            print(f"   Doc {i+1}: {'RELEVANT' if is_relevant else 'NOT RELEVANT'}")
            if is_relevant:
                relevant_docs.append(doc)

        if not relevant_docs:
            return {"answer": "No relevant information found.", "sources": []}

        # Generate
        print(f"\n3. Generating from {len(relevant_docs)} docs...")
        context = "\n---\n".join([d.page_content for d in relevant_docs])
        chain = self.generation_prompt | self.llm | StrOutputParser()
        answer = chain.invoke(
            {"context": context, "query": query},
            config={"callbacks": [self.opik_tracer]},
        )

        return {
            "answer": answer,
            "sources": [d.metadata for d in relevant_docs]
        }

self_rag = SelfRAG(retriever)
print("Self-RAG ready!")

Self-RAG ready!


## 4. Self-RAG: Success Scenario

In [7]:
# Simple query - Self-RAG succeeds
simple_query = "What is the deductible for auto insurance?"

result = self_rag.query(simple_query)

print("\n" + "="*60)
print("ANSWER:")
print("="*60)
print(result["answer"])
print(f"\nSources: {result['sources']}")

OPIK: Started logging traces to the "agentic-rag-insurance" project at https://www.comet.com/opik/api/v1/session/redirect/projects/?trace_id=019eee97-823e-70a3-97dd-dcb46d05d944&path=aHR0cHM6Ly93d3cuY29tZXQuY29tL29waWsvYXBpLw==.



SELF-RAG: What is the deductible for auto insurance?...

1. Retrieving...
   Found 3 documents

2. Checking relevance...
   Doc 1: RELEVANT
   Doc 2: NOT RELEVANT
   Doc 3: NOT RELEVANT

3. Generating from 1 docs...

ANSWER:
The policy specifies a **deductible of $500 for standard drivers and $1,000 for high‑risk drivers**【AUTO INSURANCE POLICY TERMS - COMPREHENSIVE COVERAGE】.

Sources: [{'type': 'policy', 'category': 'auto'}]


## 5. Self-RAG: Failure Scenario

In [8]:
# Complex query - Self-RAG struggles
complex_query = """
A California policyholder John Smith filed a $15,000 water damage claim
from a burst pipe. What approvals are needed and what are the regulatory deadlines?
"""

result = self_rag.query(complex_query)

print("\n" + "="*60)
print("ANSWER:")
print("="*60)
print(result["answer"])
print(f"\nSources: {result['sources']}")
print("\n** Note: Self-RAG may miss some info as it only retrieves once **")


SELF-RAG: 
A California policyholder John Smith filed a $15,...

1. Retrieving...
   Found 3 documents

2. Checking relevance...
   Doc 1: NOT RELEVANT
   Doc 2: RELEVANT
   Doc 3: RELEVANT

3. Generating from 2 docs...

ANSWER:
**Approvals needed**

- Because the claim amount is **$15,000 > $10,000**, it **requires supervisor approval** (Priority Processing Rules – “Claims over $10,000: Require supervisor approval”).  

**Documentation required**

- As a property‑damage claim, the claim must be supported by **photo evidence** (Documentation Requirements – “Photo evidence for property damage”).  
- The policy also calls for **repair estimates from two contractors** (Documentation Requirements – “Repair estimates from 2 contractors”).

**Regulatory deadlines (California)**  

| Step | Deadline | Source |
|------|----------|--------|
| Acknowledge claim receipt | **Within 15 days** of filing | STATE REGULATORY REQUIREMENTS – CALIFORNIA – “Acknowledge claim receipt: Within 15 days” |
| A

**Issues with the answer**
| # | Decomposed Question Part | Expected from KB | Answer Coverage | Correct? | Issue |
|---|--------------------------|-----------------|-----------------|----------|-------|
| 1 | Water damage source &<br>category | Burst pipe →<br>Category 1 (Clean) | Mentions Category 1<br>(Clean) | ✅ Yes | — |
| 2 | Applicable<br>procedure | Follow Water Damage<br>Special Procedures | Photos, plumber report,<br>moisture readings mentioned | ✅ Yes | — |
| 3 | Claim amount impact<br>($15,000) | Claims > $10,000 <br>require supervisor approval | Not mentioned | ❌ No | Missed mandatory<br>supervisor approval |
| 4 | Claims history<br>consideration | Multiple claims →<br>Flag for fraud review | Not mentioned | ❌ No | Claims history ignored |
| 5 | Fraud indicators | None detected in<br>claims history | Not addressed | ⚠️ Partial | Should state “flagged but<br>no indicators” |
| 6 | CA acknowledgment<br>deadline | Acknowledge claim<br>within 15 days | 15 days stated | ✅ Yes | — |
| 7 | CA accept/deny<br>deadline | Accept or deny within<br>40 days | 40 days stated | ✅ Yes | — |
| 8 | CA payment<br>deadline | Payment within 30 days<br>after acceptance | 30 days stated | ✅ Yes | — |
| 9 | Source attribution | Procedure + Guidelines +<br>Regulation + History | Home policy cited<br>unnecessarily | ❌ No | Noisy / incorrect<br>citation |


## 6. Agentic RAG Architecture

```
User Query --> Orchestrator Agent --> Multiple Tools --> Synthesize Answer
                    |                     |
                    v                     v
              - Plan steps          - Retriever
              - Select tools        - Claims DB lookup
              - Iterate             - Regulation checker
                                    - Deadline calculator
```

| Aspect | Self-RAG | Agentic RAG |
|--------|----------|-------------|
| Retrieval | Single pass | Multiple, dynamic |
| Tools | Just retriever | Multiple specialized |
| Reasoning | Linear | Iterative |

## 7. Agentic RAG Implementation

In [9]:
from datetime import datetime, timedelta
from openai import OpenAI
from opik import track
from opik.integrations.openai import track_openai
import json

# Point the OpenAI SDK at Groq's OpenAI-compatible endpoint, then wrap for Opik tracing.
# track_openai auto-logs every client.chat.completions.create() call (prompts,
# responses, model, token usage, latency) with no other change to the loop body.
client = OpenAI(base_url=GROQ_BASE_URL, api_key=os.environ["GROQ_API_KEY"])
client = track_openai(client, project_name=OPIK_PROJECT_NAME)

# Define tools as functions
# @track(type="tool") logs each tool execution as its own span (inputs + outputs),
# so the full tool-call chain is visible in the Opik trace tree.
@track(type="tool", project_name=OPIK_PROJECT_NAME)
def retrieve_documents(query: str, doc_type: str = None) -> str:
    """Retrieve documents from knowledge base"""
    results = retriever.invoke(query)
    if doc_type:
        results = [r for r in results if r.metadata.get("type") == doc_type]

    output = []
    for i, doc in enumerate(results[:3]):
        output.append(f"Doc {i+1} ({doc.metadata.get('type', 'unknown')}):")
        output.append(doc.page_content[:500])
    return "\n".join(output) if output else "No documents found"

@track(type="tool", project_name=OPIK_PROJECT_NAME)
def lookup_claims_history(name: str) -> str:
    """Look up claims history for a policyholder"""
    for doc in INSURANCE_DOCUMENTS:
        if doc["metadata"].get("type") == "claims_history":
            if name.lower() in doc["content"].lower():
                return doc["content"]
    return f"No claims history found for {name}"

@track(type="tool", project_name=OPIK_PROJECT_NAME)
def get_state_regulations(state: str) -> str:
    """Get state regulatory requirements"""
    for doc in INSURANCE_DOCUMENTS:
        if doc["metadata"].get("type") == "regulation":
            if doc["metadata"].get("state", "").upper() == state.upper():
                return doc["content"]
    return f"No regulations found for {state}"

@track(type="tool", project_name=OPIK_PROJECT_NAME)
def check_approval_requirements(amount: float) -> str:
    """Check what approvals are needed for claim amount"""
    if amount > 50000:
        return f"${amount:,.0f} claim: VP Claims approval required"
    elif amount > 10000:
        return f"${amount:,.0f} claim: Supervisor approval required"
    else:
        return f"${amount:,.0f} claim: Analyst can approve directly"

@track(type="tool", project_name=OPIK_PROJECT_NAME)
def calculate_deadline(days: int, deadline_type: str) -> str:
    """Calculate deadline from today"""
    deadline = datetime.now() + timedelta(days=days)
    return f"{deadline_type}: {deadline.strftime('%Y-%m-%d')} ({days} days from today)"

# Tool definitions for OpenAI
tools = [
    {
        "type": "function",
        "function": {
            "name": "retrieve_documents",
            "description": "Search knowledge base for policy terms, guidelines, procedures",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"},
                    "doc_type": {"type": "string", "description": "Filter by type: policy, guideline, procedure, regulation"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_claims_history",
            "description": "Look up claims history for a policyholder by name",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Policyholder name"}
                },
                "required": ["name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_state_regulations",
            "description": "Get state-specific regulatory requirements and deadlines",
            "parameters": {
                "type": "object",
                "properties": {
                    "state": {"type": "string", "description": "State abbreviation like CA, NY, TX"}
                },
                "required": ["state"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_approval_requirements",
            "description": "Check what approvals are needed based on claim dollar amount",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number", "description": "Claim amount in dollars"}
                },
                "required": ["amount"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_deadline",
            "description": "Calculate a deadline date from today",
            "parameters": {
                "type": "object",
                "properties": {
                    "days": {"type": "integer", "description": "Number of days until deadline"},
                    "deadline_type": {"type": "string", "description": "Type of deadline"}
                },
                "required": ["days", "deadline_type"]
            }
        }
    }
]

# Function to execute tools
def execute_tool(name: str, args: dict) -> str:
    if name == "retrieve_documents":
        return retrieve_documents(args.get("query", ""), args.get("doc_type"))
    elif name == "lookup_claims_history":
        return lookup_claims_history(args.get("name", ""))
    elif name == "get_state_regulations":
        return get_state_regulations(args.get("state", ""))
    elif name == "check_approval_requirements":
        return check_approval_requirements(args.get("amount", 0))
    elif name == "calculate_deadline":
        return calculate_deadline(args.get("days", 0), args.get("deadline_type", ""))
    return "Unknown tool"

print("Agentic RAG tools ready!")

Agentic RAG tools ready!


In [10]:
# @track makes the whole multi-iteration tool loop one Opik trace, with each LLM
# turn (via track_openai) and each tool call (via @track tools) nested underneath.
@track(project_name=OPIK_PROJECT_NAME)
def agentic_rag_query(query: str, verbose: bool = True) -> dict:
    """Run agentic RAG with tool calling"""

    if verbose:
        print(f"\n{'='*60}")
        print("AGENTIC RAG")
        print(f"{'='*60}")
        print(f"Query: {query[:80]}...")

    messages = [
        {
            "role": "system",
            "content": """You are an expert Claims Analyst Assistant.
Use the available tools to gather all necessary information before answering.
For complex queries, use multiple tools to get complete information.
Always cite your sources."""
        },
        {"role": "user", "content": query}
    ]

    tool_calls_made = []
    max_iterations = 10

    for iteration in range(max_iterations):
        response = client.chat.completions.create(
            model=GROQ_CHAT_MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        message = response.choices[0].message

        # If no tool calls, we have the final answer
        if not message.tool_calls:
            if verbose:
                print(f"\n--- Final answer after {len(tool_calls_made)} tool calls ---")
            return {
                "answer": message.content,
                "tool_calls": tool_calls_made
            }

        # Process tool calls
        messages.append(message)

        for tool_call in message.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)

            if verbose:
                print(f"\n-> Tool: {func_name}")
                print(f"   Args: {func_args}")

            result = execute_tool(func_name, func_args)

            if verbose:
                print(f"   Result: {result[:100]}..." if len(result) > 100 else f"   Result: {result}")

            tool_calls_made.append({"tool": func_name, "args": func_args})

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })

    return {"answer": "Max iterations reached", "tool_calls": tool_calls_made}

print("Agentic RAG query function ready!")

Agentic RAG query function ready!


## 8. Agentic RAG: Solving the Complex Query

In [11]:
# Run the complex query with Agentic RAG
complex_query = """
A California policyholder John Smith filed a $15,000 water damage claim
from a burst pipe. Given his claims history, what is the processing procedure,
what approvals are needed, and what are the regulatory deadlines?
"""

result = agentic_rag_query(complex_query)


AGENTIC RAG
Query: 
A California policyholder John Smith filed a $15,000 water damage claim
from a ...

-> Tool: retrieve_documents
   Args: {'doc_type': 'procedure', 'query': 'claims processing procedure water damage burst pipe'}
   Result: Doc 1 (procedure):
WATER DAMAGE CLAIMS - SPECIAL PROCEDURES

Step 1: Determine water source
- Catego...

-> Tool: retrieve_documents
   Args: {'doc_type': 'policy', 'query': 'approval requirements claim amount'}
   Result: No documents found

-> Tool: check_approval_requirements
   Args: {'amount': 15000}
   Result: $15,000 claim: Supervisor approval required

-> Tool: get_state_regulations
   Args: {'state': 'CA'}
   Result: STATE REGULATORY REQUIREMENTS - CALIFORNIA

Claims Processing Timelines:
- Acknowledge claim receipt...

-> Tool: lookup_claims_history
   Args: {'name': 'John Smith'}
   Result: CLAIMS HISTORY - John Smith (Policy #AC-2024-78432)

Claim 1 (2023-03-15): Fender bender, $2,340 pai...

--- Final answer after 5 tool calls ---


In [12]:
# Display final answer
print("\n" + "="*60)
print("FINAL ANSWER")
print("="*60)
print(result["answer"])


FINAL ANSWER
**1. Processing Procedure – Water‑Damage Claim (Burst Pipe)**  

| Step | What to Do | Source |
|------|------------|--------|
| **a. Initial Intake** | Log the claim, assign a water‑damage adjuster, and send an acknowledgement to the policyholder. | CA regulator – “Acknowledge claim receipt: Within 15 days”【5】 |
| **b. Determine Water Source** | Classify the source (Category 1 – clean water from a broken supply line). | Water‑damage special‑procedure “Determine water source”【1】 |
| **c. Coverage Assessment** | Verify that the loss is **sudden & accidental** (covered) and not gradual or maintenance‑related (not covered). | Water‑damage special‑procedure “Coverage assessment”【1】 |
| **d. Gather Required Documentation** | • Photos of the damage  <br>• Licensed plumber’s report confirming burst pipe  <br>• Moisture‑meter readings | Water‑damage special‑procedure “Required: Photos, plumber report, moisture readings”【1】 |
| **e. Investigation & Estimate** | Adjuster conducts o

In [13]:
# Show tool calls made
print("\n" + "="*60)
print("TOOLS USED")
print("="*60)
for i, tc in enumerate(result["tool_calls"], 1):
    print(f"{i}. {tc['tool']}: {tc['args']}")


TOOLS USED
1. retrieve_documents: {'doc_type': 'procedure', 'query': 'claims processing procedure water damage burst pipe'}
2. retrieve_documents: {'doc_type': 'policy', 'query': 'approval requirements claim amount'}
3. check_approval_requirements: {'amount': 15000}
4. get_state_regulations: {'state': 'CA'}
5. lookup_claims_history: {'name': 'John Smith'}


## 9. Comparison Summary

| Metric | Self-RAG | Agentic RAG |
|--------|----------|-------------|
| Simple Query | Success | Success |
| Complex Query | Incomplete | Comprehensive |
| Latency | ~2-3 sec | ~8-15 sec |
| Cost | Lower | Higher |
| Tool Calls | 1 | 5-8 |

### When to Use Each:

**Self-RAG**: Simple queries, cost-sensitive, low latency needed

**Agentic RAG**: Complex multi-step queries, compliance requirements, comprehensive answers needed

In [ ]:
# Setup
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Documents
INSURANCE_DOCUMENTS = [
    {"id": "policy_auto", "content": """AUTO INSURANCE POLICY - Deductible: $500 standard, $1000 high-risk. Exclusions: Racing, commercial use, intentional damage. Claim deadline: 30 days.""", "metadata": {"type": "policy", "category": "auto"}},
    {"id": "policy_home", "content": """HOME INSURANCE POLICY - Dwelling: $500,000, Personal property: $250,000. Water damage: Burst pipes covered if sudden. Flood NOT covered. Claim deadline: 60 days.""", "metadata": {"type": "policy", "category": "home"}},
    {"id": "guidelines", "content": """CLAIMS GUIDELINES - Claims over $10,000: Supervisor approval required. Claims with injuries: 48 hour processing. Multiple claims: Fraud review. Escalation: Analyst -> Senior -> Manager -> VP.""", "metadata": {"type": "guideline"}},
    {"id": "ca_regulations", "content": """CALIFORNIA REGULATIONS - Acknowledge receipt: 15 days. Accept/deny: 40 days. Payment: 30 days after acceptance. Penalties: $10,000 per violation.""", "metadata": {"type": "regulation", "state": "CA"}},
    {"id": "john_smith", "content": """CLAIMS HISTORY - John Smith (Policy #AC-2024-78432): 3 claims in 12 months ($2,340 + $450 + $1,800). Risk: Medium. Premium adjustment: +15%. Fraud indicators: None.""", "metadata": {"type": "claims_history", "policyholder": "John Smith"}},
    {"id": "water_procedure", "content": """WATER DAMAGE PROCEDURES - Step 1: Determine source (Clean/Gray/Black). Step 2: Coverage check (Sudden=covered, Gradual=NOT). Step 3: Docs needed (photos, plumber report, moisture readings). Processing: 14-21 days.""", "metadata": {"type": "procedure", "category": "water_damage"}}
]

# Groq has no embeddings endpoint, so embed locally with sentence-transformers.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
documents = [Document(page_content=doc["content"], metadata=doc["metadata"]) for doc in INSURANCE_DOCUMENTS]
vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings, collection_name="comparison_test")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Setup complete!")